In [ ]:
!pip install --upgrade pip
!pip install -U yfinance

import yfinance as yf
from yfinance import EquityQuery  # for sector/region screening
import pandas as pd
import numpy as np
import os

# =============================
# Helper: dynamic universe by sector
# =============================
def get_tickers_by_sector(
    sector: str = "Technology",
    region: str = "us",
    size: int = 20,
    blacklist: list | None = None,
) -> list:
    """
    Use Yahoo Finance screener via yfinance to get tickers
    for a given sector & region.

    sector: Yahoo sector name, e.g. "Technology", "Energy", "Healthcare"
    region: e.g. "us", "gb", "fr", ...
    size:   number of tickers to return (max 250 per request)
    blacklist: list of tickers to exclude explicitly
    """
    if blacklist is None:
        blacklist = []

    # Build query: sector == X AND region == Y
    q = EquityQuery("and", [
        EquityQuery("eq", ["sector", sector]),
        EquityQuery("eq", ["region", region]),
    ])

    # Run screen
    resp = yf.screen(q, size=size)

    quotes = resp.get("quotes", [])
    if not quotes:
        print(f"No quotes returned for sector={sector}, region={region}")
        return []

    df = pd.DataFrame(quotes)

    if "symbol" not in df.columns:
        raise ValueError(f"Unexpected screener response columns: {df.columns}")

    tickers = df["symbol"].dropna().astype(str).tolist()

    # Simple quality filters
    #  - drop warrants/rights often ending with 'W'
    #  - apply explicit blacklist (e.g. ZEOWW)
    tickers = [t for t in tickers if not t.endswith("W")]
    tickers = [t for t in tickers if t not in set(blacklist)]

    print(f"Found {len(tickers)} usable tickers for sector={sector}, region={region}")
    return tickers


# =============================
# Market Data Handler
# =============================
class MarketDataHandler:
    def __init__(self, tickers, start, end, save_path="market_data.xlsx"):
        self.tickers = tickers
        self.start = start
        self.end = end
        self.save_path = save_path
        self.data = None
        self.returns = None

    def load_data(self):
        raw = yf.download(self.tickers, start=self.start, end=self.end, auto_adjust=True)
        self.data = raw["Close"].copy()
        print(f"Loaded {len(self.data)} days of data for {len(self.tickers)} tickers.")
        return self.data

    def clean_data(self):
        # 1) Drop tickers (columns) that are all NaN (e.g., bad/delisted ones)
        self.data = self.data.dropna(axis=1, how="all")

        # 2) Drop dates (rows) that are completely empty
        self.data = self.data.dropna(axis=0, how="all").astype(float)

        print(f"Cleaned data — shape: {self.data.shape}")
        print("Remaining tickers:", list(self.data.columns))

        # Optional: save cleaned prices
        self.data.to_excel(self.save_path)
        print(f"Market data saved to {self.save_path}")
        return self.data

    def compute_returns(self):
        self.returns = self.data.pct_change().dropna()
        self.returns.to_excel("returns_data.xlsx")
        print("Saved returns_data.xlsx")
        return self.returns


# =============================
# Pipeline Execution
# =============================
if __name__ == "__main__":
    # Pick sector/region & universe size instead of hard-coding tickers
    sector = "Technology"   # e.g. "Technology", "Energy", "Healthcare", ...
    region = "us"           # e.g. "us", "gb", "fr", ...
    universe_size = 20      # how many tickers you want

    # Optional: blacklist known bad tickers if they ever show up
    blacklist = ["ZEOWW"]

    tickers = get_tickers_by_sector(
        sector=sector,
        region=region,
        size=universe_size,
        blacklist=blacklist,
    )
    print("Selected tickers:", tickers)

    if not tickers:
        raise SystemExit("No tickers found – adjust sector/region/universe_size.")

    # Save selected universe to Excel so you can see which companies you got
    pd.DataFrame({"ticker": tickers}).to_excel("selected_universe.xlsx", index=False)
    print("Saved selected_universe.xlsx")

    # Step 1: Market data
    market = MarketDataHandler(tickers, "2020-01-01", "2025-01-01")
    prices = market.load_data()
    clean_prices = market.clean_data()

    # Step 2: Returns
    returns = market.compute_returns()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Found 19 usable tickers for sector=Technology, region=us
Selected tickers: ['ZVTK', 'ZVLO', 'ZTSTF', 'ZTNO', 'ZTLLF', 'ZTCOF', 'ZSTN', 'ZSPC', 'ZS', 'ZRFY', 'ZPTA', 'ZPAS', 'ZNAE', 'ZM', 'ZICXD', 'ZICX', 'ZEUCF', 'ZETA', 'ZEPP']
Saved selected_universe.xlsx


[*********************100%***********************]  19 of 19 completed


Loaded 1258 days of data for 19 tickers.
Cleaned data — shape: (1258, 19)
Remaining tickers: ['ZEPP', 'ZETA', 'ZEUCF', 'ZICX', 'ZICXD', 'ZM', 'ZNAE', 'ZPAS', 'ZPTA', 'ZRFY', 'ZS', 'ZSPC', 'ZSTN', 'ZTCOF', 'ZTLLF', 'ZTNO', 'ZTSTF', 'ZVLO', 'ZVTK']
Market data saved to market_data.xlsx
Saved returns_data.xlsx
